# RPython on Google Colab / Jupyter — quickstart

Fresh runtime → install → setup → R package → data transfer → plot → magics → diagnostics.

On Colab uncomment the pip line. Locally, the package is already installed.

In [ ]:
# !pip install "rpython[arrow]"
import rpython as rp, pandas as pd, numpy as np
r = rp.setup()   # detects R, starts the worker, loads the %r / %%r magics
r

## Environment doctor

In [ ]:
rep = rp.doctor()
assert rep.ok or True  # a warning-only report is fine on CI runners

## Install and use an R package

In [ ]:
print(r.installed('stats'))
stats = r.package('stats')
print(stats.median([1, 2, 3, 10]))

## Send a pandas DataFrame, get an R model back

In [ ]:
df = pd.DataFrame({'x': np.arange(20.), 'g': pd.Categorical(list('ab') * 10, ordered=True)})
df['y'] = 2 * df['x'] + (df['g'] == 'b') + np.random.default_rng(0).normal(size=20)
r['df'] = df
fit = r('lm(y ~ x + g, data = df)')
fit.coef()

## Plots display inline

In [ ]:
res = r.eval('plot(df$x, df$y); abline(lm(y ~ x, data = df))')
res.plot

## Magics

In [ ]:
%%r -i df -o coefs
fit2 <- lm(y ~ x, data = df)
coefs <- coef(fit2)
summary(fit2)$r.squared

In [ ]:
coefs

## Explain Mode and self-test

In [ ]:
rp.explain_last()
rp.self_test()

## Reproducibility after a runtime reset

`rpython lock` writes `rpython.lock`; `rp.restore('rpython.lock')` reinstalls missing packages on a fresh runtime.

In [ ]:
from rpython.env.lock import lock
lock('rpython.lock')